# LeNet IMC + ADC accuracy sweep (GPU)

Hybrid SRAM/PCM-IMC accuracy experiment for LeNet on a Colab GPU.
Arms: no-protection / uniform / layer-aware, each at ADC off and 5-bit, over a drift-time grid, multi-seed.

**You only upload TWO files in total:**
1. this notebook (already open), and
2. `data.zip` (a zip of your `rtl_inputs_A` and `rtl_inputs_A_v3` folders), uploaded in step 4 below.

The three Python files are written to disk by cell 2 (no need to upload them).

**Before running:** Runtime -> Change runtime type -> T4 GPU.


## 1. Check GPU and torch

In [1]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - set Runtime to T4 GPU')

torch 2.11.0+cu128 | cuda available: True
Tesla T4


## 2. Write the three Python files to disk
Run all three cells (they just create the .py files in Colab).

In [3]:
%%writefile imc_vec.py
#!/usr/bin/env python3
"""
imc_vec.py  --  vectorized (numpy) INT8 hybrid SRAM/PCM-IMC forward for LeNet.

Same compute model as imc_forward_v3_imc.py (the per-element scaffold), but
batched over images so it can run a test set. Must reproduce the scaffold:
  - ADC off + drift off  => bit-exact vs the RTL golden
  - ADC 5-bit + drift off => identical fc3 logits to the scaffold

Layer math (after im2col, conv and fc are both matmuls out[o,m]=sum_k W[o,k]V[k,m]):
  protected (mask True)  -> exact integer partial (SRAM path)
  unprotected (mask False)-> PCM-IMC:
        |w| -> cellBit=2 cells C_j=(|w|>>2j)&3 ; sign via +/- polarity
        v (unsigned INT8) -> bit-serial b_i=(v>>i)&1
        rows tiled by subArray=64
        per (i,j,polarity,tile): analog partial = sum_row b_i*(C_j*drift)
            optional 5-bit ADC over [0, actual_rows_in_tile * cmax]
        shift-add: total += 2^i * 2^(2j) * (ADC(pos)-ADC(neg))
        drift: per-weight t^(-nu/256) on that weight's cells (off: t=1)
        compensation t^(+comp) applied to total AFTER ADC (off: comp=0)
  acc = bias + protected + pcm ; requant = arsh(acc,rshift) + relu/sat
"""

import json
import math
import os
import numpy as np

FFFF = 0xFFFF
LAYERS = ["conv1", "conv2", "fc1", "fc2", "fc3"]
WCOUNT = {"conv1": 150, "conv2": 2400, "fc1": 48000, "fc2": 10080, "fc3": 840}

DEFAULT_CFG = {
    "cellBit": 2, "n_mag_cells": 4, "subArray": 64, "in_bits": 8,
    "adc_bits": None, "drift_t": 1.0, "comp": 0.0,
}


# ---------- hex readers ----------
def read_hex_bytes(path, signed=True):
    out = []
    with open(path) as f:
        for ln in f:
            ln = ln.strip()
            if not ln or ln.startswith("//"):
                continue
            v = int(ln, 16)
            if signed and v >= 128:
                v -= 256
            out.append(v)
    return out


def read_hex_words(path, width_bits=32):
    out = []
    sb = 1 << (width_bits - 1)
    full = 1 << width_bits
    with open(path) as f:
        for ln in f:
            ln = ln.strip()
            if not ln or ln.startswith("//"):
                continue
            v = int(ln, 16)
            if v >= sb:
                v -= full
            out.append(v)
    return out


def read_hex_u(path):
    out = []
    with open(path) as f:
        for ln in f:
            ln = ln.strip()
            if not ln or ln.startswith("//"):
                continue
            out.append(int(ln, 16))
    return out


# ---------- model loading ----------
def load_model(src_dir, v3_dir):
    """Returns dict with per-layer W (out,K) int, mask (out,K) bool protected,
    nu (out,K) int code, plus biases and rshift."""
    shapes = {
        "conv1": (6, 1 * 5 * 5), "conv2": (16, 6 * 5 * 5),
        "fc1": (120, 400), "fc2": (84, 120), "fc3": (10, 84),
    }
    raw = {L: np.array(read_hex_bytes(os.path.join(src_dir, f"weights_{L}.hex")),
                        dtype=np.int64) for L in LAYERS}
    for L in LAYERS:
        assert raw[L].size == WCOUNT[L]

    with open(os.path.join(v3_dir, "split_summary_v3.json")) as f:
        summ = json.load(f)
    sram_pool = np.array(read_hex_bytes(os.path.join(v3_dir, "sram_pool.hex")), dtype=np.int64)
    pcm_pool = np.array(read_hex_bytes(os.path.join(v3_dir, "pcm_pool.hex")), dtype=np.int64)
    nu_pool = np.array(read_hex_u(os.path.join(v3_dir, "nu_pcm_pool.hex")), dtype=np.int64)

    W, mask, nu = {}, {}, {}
    recon_bad = 0
    for L in LAYERS:
        n = WCOUNT[L]
        a_sram = np.array(read_hex_u(os.path.join(v3_dir, f"addr_to_slot_{L}_sram.hex")), dtype=np.int64)
        a_pcm = np.array(read_hex_u(os.path.join(v3_dir, f"addr_to_slot_{L}_pcm.hex")), dtype=np.int64)
        sbase = summ["layers"][L]["sram_base"]
        pbase = summ["layers"][L]["pcm_base"]
        m_flat = a_sram != FFFF
        nu_flat = np.zeros(n, dtype=np.float64)
        idx_pcm = np.where(~m_flat)[0]
        nu_flat[idx_pcm] = nu_pool[pbase + a_pcm[idx_pcm]] / 256.0   # code -> physical nu
        # reconstruction check
        rec = np.empty(n, dtype=np.int64)
        idx_sram = np.where(m_flat)[0]
        rec[idx_sram] = sram_pool[sbase + a_sram[idx_sram]]
        rec[idx_pcm] = pcm_pool[pbase + a_pcm[idx_pcm]]
        recon_bad += int(np.sum(rec != raw[L]))
        oc, K = shapes[L]
        W[L] = raw[L].reshape(oc, K)
        mask[L] = m_flat.reshape(oc, K)
        nu[L] = nu_flat.reshape(oc, K)
    assert recon_bad == 0, f"reconstruction mismatch ({recon_bad})"

    biases_all = read_hex_words(os.path.join(src_dir, "biases_all.hex"))
    rshift_all = read_hex_words(os.path.join(src_dir, "rshift_per_layer.hex"), width_bits=8)
    b = {"conv1": biases_all[0:6], "conv2": biases_all[6:22], "fc1": biases_all[22:142],
         "fc2": biases_all[142:226], "fc3": biases_all[226:236]}
    bias = {L: np.array(b[L], dtype=np.int64) for L in LAYERS}
    rshift = dict(zip(LAYERS, rshift_all[0:5]))
    return {"W": W, "mask": mask, "nu": nu, "bias": bias, "rshift": rshift}


# ---------- im2col ----------
def im2col(x, kh, kw, pad):
    """x: (B,C,H,W) -> (B, C*kh*kw, OH*OW), fan-in index = ic*kh*kw+ky*kw+kx."""
    B, C, H, W = x.shape
    OH, OW = H + 2 * pad - kh + 1, W + 2 * pad - kw + 1
    if pad:
        xp = np.zeros((B, C, H + 2 * pad, W + 2 * pad), dtype=x.dtype)
        xp[:, :, pad:pad + H, pad:pad + W] = x
    else:
        xp = x
    cols = np.empty((B, C, kh, kw, OH, OW), dtype=x.dtype)
    for ky in range(kh):
        for kx in range(kw):
            cols[:, :, ky, kx, :, :] = xp[:, :, ky:ky + OH, kx:kx + OW]
    return cols.reshape(B, C * kh * kw, OH * OW), OH, OW


# ---------- IMC matmul ----------
def imc_matmul(W, mask, nu, V, rshift, bias, cfg, kind, wn=None):
    """
    W,mask,nu: (out, K).  V: (K, M) int (M = positions*batch), unsigned for PCM.
    wn: (out, K, ncell) per-cell write-noise realization (float) or None.
    Returns requantized output (out, M) as int.
    """
    out_dim, K = W.shape
    cellBit, ncell = cfg["cellBit"], cfg["n_mag_cells"]
    sub, in_bits = cfg["subArray"], cfg["in_bits"]
    adc_bits, t, comp = cfg["adc_bits"], cfg["drift_t"], cfg["comp"]
    cmax = (1 << cellBit) - 1
    M = V.shape[1]

    # protected exact partial (integer)
    Wp = np.where(mask, W, 0)
    acc = bias[:, None] + Wp @ V                      # (out, M) int64

    # PCM path over unprotected weights
    Wu = np.where(mask, 0, W)
    if np.any(Wu != 0) or True:
        absU = np.abs(Wu)
        signpos = (Wu > 0).astype(np.float64)         # (out,K)
        signneg = (Wu < 0).astype(np.float64)
        if t != 1.0:
            D = np.where(mask, 1.0, t ** (-nu))        # nu is physical drift exponent
        else:
            D = np.ones_like(Wu, dtype=np.float64)

        # tiling: pad K to multiple of sub
        T = (K + sub - 1) // sub
        Kpad = T * sub
        actual_rows = np.full(T, sub, dtype=np.int64)
        if Kpad != K:
            actual_rows[-1] = K - (T - 1) * sub

        def padK(a, fill=0.0):
            if Kpad == K:
                return a
            out = np.full(a.shape[:-1] + (Kpad,), fill, dtype=a.dtype)
            out[..., :K] = a
            return out

        # per cell-plane: cell value scaled by drift, split by polarity, padded & tiled
        # CWpos[j] : (out, T, sub) ; same for neg
        Vf = V.astype(np.float64)                      # (K, M)
        # bit-planes of V: (in_bits, Kpad, M)
        Vp = padK(Vf.T).T if False else None           # placeholder
        # build padded V (Kpad, M)
        Vpad = np.zeros((Kpad, M), dtype=np.float64)
        Vpad[:K, :] = Vf
        Vbits = np.stack([((Vpad.astype(np.int64) >> i) & 1).astype(np.float64)
                          for i in range(in_bits)], axis=0)   # (in_bits, Kpad, M)
        Vbits_t = Vbits.reshape(in_bits, T, sub, M)            # (i, T, sub, M)

        pcm = np.zeros((out_dim, M), dtype=np.float64)
        for j in range(ncell):
            Cj = ((absU >> (cellBit * j)) & cmax).astype(np.float64)  # (out,K)
            if wn is not None:
                # per-cell write noise (programming error), unprotected cells only;
                # protected cells stay exact (SRAM). nonzero-weight polarity columns
                # carry the noise; exact-zero unprotected weights have no polarity so
                # their noise is not modeled here (secondary, see notes).
                Cj = Cj + wn[:, :, j] * (~mask)
            CjD_pos = padK(Cj * D * signpos).reshape(out_dim, T, sub)
            CjD_neg = padK(Cj * D * signneg).reshape(out_dim, T, sub)
            wj = float(1 << (cellBit * j))
            for i in range(in_bits):
                Bi = Vbits_t[i]                                   # (T, sub, M)
                # partial[out,T,M] = sum_sub Bi[T,sub,M] * CjD[out,T,sub]
                ppos = np.einsum('trm,otr->otm', Bi, CjD_pos, optimize=True)
                pneg = np.einsum('trm,otr->otm', Bi, CjD_neg, optimize=True)
                if adc_bits is not None:
                    fs = (actual_rows * cmax).astype(np.float64)  # (T,)
                    ppos = adc_quant(ppos, fs[None, :, None], adc_bits)
                    pneg = adc_quant(pneg, fs[None, :, None], adc_bits)
                contrib = (ppos - pneg).sum(axis=1)               # sum over T -> (out, M)
                bw = -(1 << i) if i == in_bits - 1 else (1 << i)   # two's-complement MSB
                pcm += (float(bw) * wj) * contrib
        if comp != 0.0:
            pcm *= (t ** comp)
        acc = acc.astype(np.float64) + pcm

    # requant
    if kind == "fc3":
        return requant_fc3(acc, rshift)
    return requant_relu(acc, rshift)


def adc_quant(x, full_scale, bits):
    levels = (1 << bits) - 1
    fs = np.asarray(full_scale, dtype=np.float64)
    q = np.where(fs > 0, np.round(x / np.where(fs > 0, fs, 1.0) * levels), 0.0)
    q = np.clip(q, 0, levels)
    return q / levels * fs


def _floor_div_shift(acc, rshift):
    # arithmetic shift / floor div toward -inf, works for int or float arrays
    return np.floor(acc / (1 << rshift))


def requant_relu(acc, rshift):
    shifted = _floor_div_shift(acc, rshift)
    out = np.where(acc <= 0, 0, np.minimum(127, shifted))
    return out.astype(np.int64)


def requant_fc3(acc, rshift):
    shifted = _floor_div_shift(acc, rshift).astype(np.int64)
    POS, NEG = 32767, -131071
    res = np.empty_like(shifted)
    pos_sat = shifted > POS
    neg_sat = shifted < NEG
    mid = ~(pos_sat | neg_sat)
    res[pos_sat] = 32767
    res[neg_sat] = -32768
    tr = (shifted[mid] & 0xFFFF)
    tr = np.where(tr & 0x8000, tr - 0x10000, tr)
    res[mid] = tr
    return res


# ---------- full LeNet forward (batched) ----------
def forward_batch(images_int, model, cfg):
    """images_int: (B, 784) signed INT8.  Returns (logits (B,10), preds (B,))."""
    B = images_int.shape[0]
    W, mask, nu, bias, rs = model["W"], model["mask"], model["nu"], model["bias"], model["rshift"]
    wn = model.get("wn")
    gw = lambda L: (wn[L] if wn is not None else None)

    x = images_int.reshape(B, 1, 28, 28)
    # conv1 (pad 2): all-protected -> exact regardless of cfg
    col, OH, OW = im2col(x, 5, 5, 2)                 # (B, 25, 784)
    V = np.transpose(col, (1, 0, 2)).reshape(25, B * OH * OW)
    c1 = imc_matmul(W["conv1"], mask["conv1"], nu["conv1"], V, rs["conv1"], bias["conv1"], cfg, "relu", gw("conv1"))
    c1 = c1.reshape(6, B, OH * OW).transpose(1, 0, 2).reshape(B, 6, 28, 28)

    p1 = c1.reshape(B, 6, 14, 2, 14, 2).max(axis=(3, 5))   # (B,6,14,14)

    col, OH, OW = im2col(p1, 5, 5, 0)               # (B,150,100)
    V = np.transpose(col, (1, 0, 2)).reshape(150, B * OH * OW)
    c2 = imc_matmul(W["conv2"], mask["conv2"], nu["conv2"], V, rs["conv2"], bias["conv2"], cfg, "relu", gw("conv2"))
    c2 = c2.reshape(16, B, OH * OW).transpose(1, 0, 2).reshape(B, 16, 10, 10)

    p2 = c2.reshape(B, 16, 5, 2, 5, 2).max(axis=(3, 5))    # (B,16,5,5)
    flat = p2.reshape(B, 400)

    f1 = imc_matmul(W["fc1"], mask["fc1"], nu["fc1"], flat.T, rs["fc1"], bias["fc1"], cfg, "relu", gw("fc1")).T
    f2 = imc_matmul(W["fc2"], mask["fc2"], nu["fc2"], f1.T, rs["fc2"], bias["fc2"], cfg, "relu", gw("fc2")).T
    f3 = imc_matmul(W["fc3"], mask["fc3"], nu["fc3"], f2.T, rs["fc3"], bias["fc3"], cfg, "fc3", gw("fc3")).T
    preds = np.argmax(f3, axis=1)
    return f3, preds


Overwriting imc_vec.py


In [4]:
%%writefile imc_torch.py
#!/usr/bin/env python3
"""
imc_torch.py -- torch (GPU-capable) port of imc_vec.py's INT8 hybrid SRAM/PCM-IMC
forward for LeNet. Same math as the numpy reference (imc_vec.py); runs on cuda for
speed on Colab. Validated to reproduce imc_vec predictions image-for-image, and to
match the RTL golden bit-exactly at the no-drift/no-ADC/no-noise point.

dtype: float64 reproduces numpy exactly; float32 is fast on GPU and is exact for
LeNet layer sizes (all integer accumulations stay below 2^24). For deeper nets
(ImageNet) use float64 or int accumulation.
"""

import math
import torch
import numpy as np
from imc_vec import load_model, read_hex_bytes, LAYERS, WCOUNT  # reuse numpy model loader

DEFAULT_CFG = {
    "cellBit": 2, "n_mag_cells": 4, "subArray": 64, "in_bits": 8,
    "adc_bits": None, "drift_t": 1.0, "comp": 0.0,
}


# ---------- move model to torch + precompute cells ----------
def to_torch_model(model, device="cpu", dtype=torch.float32, cellBit=2, ncell=4):
    cmax = (1 << cellBit) - 1
    tm = {"rshift": model["rshift"], "W": {}, "mask": {}, "nu": {}, "bias": {}, "cells": {}}
    for L in LAYERS:
        W = model["W"][L]                                  # numpy int (out,K)
        absW = np.abs(W)
        cells = np.stack([((absW >> (cellBit * j)) & cmax) for j in range(ncell)], axis=-1)
        tm["W"][L] = torch.tensor(W, device=device, dtype=dtype)
        tm["mask"][L] = torch.tensor(model["mask"][L], device=device)        # bool
        tm["nu"][L] = torch.tensor(model["nu"][L], device=device, dtype=dtype)
        tm["bias"][L] = torch.tensor(model["bias"][L], device=device, dtype=dtype)
        tm["cells"][L] = torch.tensor(cells, device=device, dtype=dtype)     # (out,K,ncell)
    return tm


def writenoise_torch(model, seed, device="cpu", dtype=torch.float32, cellBit=2, ncell=4):
    sigma_value = 0.03 * 127.0
    S = sum((2 ** (cellBit * j)) ** 2 for j in range(ncell))
    sc = sigma_value / math.sqrt(S)
    rng = np.random.RandomState(seed)
    return {L: torch.tensor(rng.normal(0.0, sc, model["W"][L].shape + (ncell,)),
                            device=device, dtype=dtype) for L in LAYERS}


def nu_torch(model, seed, device="cpu", dtype=torch.float32):
    rng = np.random.RandomState(seed)
    return {L: torch.tensor(rng.normal(0.10, 0.03, model["W"][L].shape),
                            device=device, dtype=dtype) for L in LAYERS}


# ---------- helpers ----------
def adc_quant_t(x, fs, bits):
    levels = (1 << bits) - 1
    safe = torch.where(fs > 0, fs, torch.ones_like(fs))
    q = torch.where(fs > 0, torch.round(x / safe * levels), torch.zeros_like(x))
    q = torch.clamp(q, 0, levels)
    return q / levels * fs


def requant_relu_t(acc, rshift):
    shifted = torch.floor(acc / (1 << rshift))
    return torch.where(acc <= 0, torch.zeros_like(acc), torch.clamp(shifted, max=127))


def requant_fc3_t(acc, rshift):
    shifted = torch.floor(acc / (1 << rshift))
    POS, NEG = 32767, -131071
    tr = shifted.to(torch.int64) & 0xFFFF
    tr = torch.where((tr & 0x8000) != 0, tr - 0x10000, tr).to(acc.dtype)
    res = torch.where(shifted > POS, torch.full_like(shifted, 32767.0),
                      torch.where(shifted < NEG, torch.full_like(shifted, -32768.0), tr))
    return res


def im2col_t(x, kh, kw, pad):
    B, C, H, W = x.shape
    OH, OW = H + 2 * pad - kh + 1, W + 2 * pad - kw + 1
    if pad:
        xp = torch.zeros((B, C, H + 2 * pad, W + 2 * pad), device=x.device, dtype=x.dtype)
        xp[:, :, pad:pad + H, pad:pad + W] = x
    else:
        xp = x
    cols = torch.empty((B, C, kh, kw, OH, OW), device=x.device, dtype=x.dtype)
    for ky in range(kh):
        for kx in range(kw):
            cols[:, :, ky, kx, :, :] = xp[:, :, ky:ky + OH, kx:kx + OW]
    return cols.reshape(B, C * kh * kw, OH * OW), OH, OW


# ---------- IMC matmul (torch) ----------
def imc_matmul_t(tm, L, V, cfg, kind, wn_L=None):
    cells_L, W_L, mask_L, nu_L, bias_L = tm["cells"][L], tm["W"][L], tm["mask"][L], tm["nu"][L], tm["bias"][L]
    rshift = tm["rshift"][L]
    device, dtype = W_L.device, W_L.dtype
    out_dim, K = W_L.shape
    cellBit, ncell, sub, in_bits = cfg["cellBit"], cfg["n_mag_cells"], cfg["subArray"], cfg["in_bits"]
    adc_bits, t, comp = cfg["adc_bits"], cfg["drift_t"], cfg["comp"]
    cmax = (1 << cellBit) - 1
    M = V.shape[1]
    zeroW = torch.zeros_like(W_L)

    Wp = torch.where(mask_L, W_L, zeroW)
    acc = bias_L[:, None] + Wp @ V

    Wu = torch.where(mask_L, zeroW, W_L)
    if not bool((Wu != 0).any()):
        # fully protected layer (e.g. conv1): no PCM contribution, skip the analog path
        if kind == "fc3":
            return requant_fc3_t(acc, rshift)
        return requant_relu_t(acc, rshift)
    signpos = (Wu > 0).to(dtype)
    signneg = (Wu < 0).to(dtype)
    if t != 1.0:
        Dpcm = torch.pow(torch.as_tensor(float(t), device=device, dtype=dtype), -nu_L)
        D = torch.where(mask_L, torch.ones_like(nu_L), Dpcm)
    else:
        D = torch.ones_like(W_L)

    T = (K + sub - 1) // sub
    Kpad = T * sub
    actual_rows = torch.full((T,), float(sub), device=device, dtype=dtype)
    if Kpad != K:
        actual_rows[-1] = float(K - (T - 1) * sub)

    def padK(a):
        if Kpad == K:
            return a
        o = torch.zeros(a.shape[:-1] + (Kpad,), device=device, dtype=dtype)
        o[..., :K] = a
        return o

    Vpad = torch.zeros((Kpad, M), device=device, dtype=dtype)
    Vpad[:K, :] = V
    Vint = Vpad.to(torch.int64)
    Vbits = torch.stack([((Vint >> i) & 1).to(dtype) for i in range(in_bits)], dim=0)
    Vbits_t = Vbits.reshape(in_bits, T, sub, M)

    notmask = (~mask_L).to(dtype)
    pcm = torch.zeros((out_dim, M), device=device, dtype=dtype)
    for j in range(ncell):
        Cj = cells_L[:, :, j]
        if wn_L is not None:
            Cj = Cj + wn_L[:, :, j] * notmask
        CjD_pos = padK(Cj * D * signpos).reshape(out_dim, T, sub)
        CjD_neg = padK(Cj * D * signneg).reshape(out_dim, T, sub)
        wj = float(1 << (cellBit * j))
        for i in range(in_bits):
            Bi = Vbits_t[i]
            ppos = torch.einsum('trm,otr->otm', Bi, CjD_pos)
            pneg = torch.einsum('trm,otr->otm', Bi, CjD_neg)
            if adc_bits is not None:
                fs = (actual_rows * cmax)
                ppos = adc_quant_t(ppos, fs[None, :, None], adc_bits)
                pneg = adc_quant_t(pneg, fs[None, :, None], adc_bits)
            contrib = (ppos - pneg).sum(dim=1)
            bw = -(1 << i) if i == in_bits - 1 else (1 << i)
            pcm = pcm + (float(bw) * wj) * contrib
    if comp != 0.0:
        pcm = pcm * (math.exp(comp * math.log(t)) if t > 0 else 1.0)
    acc = acc + pcm

    if kind == "fc3":
        return requant_fc3_t(acc, rshift)
    return requant_relu_t(acc, rshift)


# ---------- full LeNet forward (torch, batched) ----------
def forward_batch_t(images_int, tm, cfg, wn=None):
    device, dtype = tm["W"]["conv1"].device, tm["W"]["conv1"].dtype
    B = images_int.shape[0]
    gw = lambda L: (wn[L] if wn is not None else None)
    x = torch.tensor(images_int.reshape(B, 1, 28, 28), device=device, dtype=dtype)

    col, OH, OW = im2col_t(x, 5, 5, 2)
    V = col.permute(1, 0, 2).reshape(25, B * OH * OW)
    c1 = imc_matmul_t(tm, "conv1", V, cfg, "relu", gw("conv1"))
    c1 = c1.reshape(6, B, OH * OW).permute(1, 0, 2).reshape(B, 6, 28, 28)
    p1 = c1.reshape(B, 6, 14, 2, 14, 2).amax(dim=(3, 5))

    col, OH, OW = im2col_t(p1, 5, 5, 0)
    V = col.permute(1, 0, 2).reshape(150, B * OH * OW)
    c2 = imc_matmul_t(tm, "conv2", V, cfg, "relu", gw("conv2"))
    c2 = c2.reshape(16, B, OH * OW).permute(1, 0, 2).reshape(B, 16, 10, 10)
    p2 = c2.reshape(B, 16, 5, 2, 5, 2).amax(dim=(3, 5))
    flat = p2.reshape(B, 400)

    f1 = imc_matmul_t(tm, "fc1", flat.t().contiguous(), cfg, "relu", gw("fc1")).t()
    f2 = imc_matmul_t(tm, "fc2", f1.t().contiguous(), cfg, "relu", gw("fc2")).t()
    f3 = imc_matmul_t(tm, "fc3", f2.t().contiguous(), cfg, "fc3", gw("fc3")).t()
    preds = torch.argmax(f3, dim=1)
    return f3, preds


Writing imc_torch.py


In [5]:
%%writefile run_lenet_gpu.py
#!/usr/bin/env python3
"""
run_lenet_gpu.py -- LeNet INT8 hybrid SRAM/PCM-IMC accuracy sweep on GPU (torch).

Same experiment as run_lenet_full.py but uses imc_torch (cuda) so the full
10000-image x multi-seed x drift-grid sweep runs in minutes on Colab instead of
hours on CPU. Forward is validated to reproduce the numpy reference image-for-image
and the RTL golden bit-exactly at the no-drift/no-ADC/no-noise point.

Arms (per drift time t, per seed): algo_only / uniform / layer_aware, each at ADC
off and 5-bit. clean = pristine INT8 (no drift, no ADC, no write noise).
Non-idealities: drift t^(-nu) (nu ~ N(0.10,0.03), Paper 1); per-cell write noise
(sigma matched to Paper 1 WRITE_NOISE=0.03); 5-bit ADC (Param.cpp levelOutput=32).
cellBit=2 sign-magnitude (NeuroSim Scheme III).

Inputs: --mnist (torchvision, calibrates exact input scale, self-checks vs
img_001.hex) or --hex-dir (pre-quantized img_*.hex).
"""

import argparse
import json
import math
import os
import re
import time
import numpy as np
import torch

from imc_vec import load_model, read_hex_bytes, LAYERS
from imc_torch import to_torch_model, forward_batch_t, nu_torch, writenoise_torch, DEFAULT_CFG

COMP = 0.10


# ---------- masks (numpy, Paper 1 semantics) ----------
def make_uniform_mask(model, budget=0.05):
    return {L: (np.abs(model["W"][L]) >= np.percentile(np.abs(model["W"][L]), 100 * (1 - budget)))
            for L in LAYERS}


def masks_to_torch(maskdict, device):
    return {L: torch.tensor(maskdict[L], device=device) for L in LAYERS}


def writenoise_sigma_cell(cellBit=2, ncell=4):
    return 0.03 * 127.0 / math.sqrt(sum((2 ** (cellBit * j)) ** 2 for j in range(ncell)))


# ---------- input loading ----------
def read_input_scale(src_dir):
    for ln in open(os.path.join(src_dir, "scales.txt")):
        m = re.match(r"\s*input\s+([\d.eE+-]+)", ln)
        if m:
            return float(m.group(1))
    raise SystemExit("input scale not found")


def calibrate_input_scale(root, n_calib=1000):
    from torchvision import datasets, transforms
    tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    ds = datasets.MNIST(root, train=True, download=True, transform=tf)
    mx = 0.0
    for i in range(n_calib):
        x, _ = ds[i]
        mx = max(mx, float(x.abs().max()))
    return mx / 127.0


def load_images_mnist(n, input_scale, root, selfcheck_hex=None):
    from torchvision import datasets, transforms
    tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    ds = datasets.MNIST(root, train=False, download=True, transform=tf)
    n = n or len(ds)
    X = np.zeros((n, 784), dtype=np.int64)
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        x, lab = ds[i]
        X[i] = np.round(x.numpy() / input_scale).clip(-128, 127).astype(np.int64).flatten()
        y[i] = int(lab)
    if selfcheck_hex and os.path.exists(selfcheck_hex):
        ref = np.array(read_hex_bytes(selfcheck_hex, signed=True))
        ok = np.array_equal(X[0], ref)
        print(f"  input self-check (test[0] vs img_001.hex): {'PASS' if ok else 'FAIL'}")
        if not ok:
            raise SystemExit("input quantization mismatch")
    return X, y


def load_images_hex(img_dir, label_file, n):
    lab = {}
    for ln in open(label_file):
        m = re.match(r"\s*img_(\d+)\s+(\d+)", ln)
        if m:
            lab[int(m.group(1))] = int(m.group(2))
    avail = sorted(int(re.match(r"img_(\d+)\.hex", f).group(1))
                   for f in os.listdir(img_dir) if re.match(r"img_\d+\.hex", f))
    avail = avail[:n] if n else avail
    X = np.array([read_hex_bytes(os.path.join(img_dir, f"img_{k:03d}.hex"), signed=True)
                  for k in avail], dtype=np.int64)
    y = np.array([lab[k] for k in avail])
    return X, y


# ---------- chunked arm runner (GPU) ----------
def run_arm(tm, arm_mask_t, X, y, nu, wn, drift_t, comp, adc_bits, chunk):
    for L in LAYERS:
        tm["mask"][L] = arm_mask_t[L]
    tm["nu"] = nu
    cfg = dict(DEFAULT_CFG)
    cfg["drift_t"] = drift_t; cfg["comp"] = comp; cfg["adc_bits"] = adc_bits
    correct = 0
    with torch.no_grad():
        for i in range(0, len(X), chunk):
            _, preds = forward_batch_t(X[i:i + chunk], tm, cfg, wn=wn)
            correct += int((preds.cpu().numpy() == y[i:i + chunk]).sum())
    return correct / len(X)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src-dir", required=True)
    ap.add_argument("--v3-dir", required=True)
    ap.add_argument("--hex-dir", default=None)
    ap.add_argument("--mnist", action="store_true")
    ap.add_argument("--mnist-root", default="./_mnist")
    ap.add_argument("--n-calib", type=int, default=1000)
    ap.add_argument("--n-images", type=int, default=0)
    ap.add_argument("--seeds", type=int, default=5)
    ap.add_argument("--drift-times", default="1,1e1,1e2,1e3,1e4,1e5,1e6")
    ap.add_argument("--adc-bits", type=int, default=5)
    ap.add_argument("--no-write-noise", action="store_true")
    ap.add_argument("--chunk", type=int, default=1000)
    ap.add_argument("--fp64", action="store_true", help="use float64 (exact, slower on GPU)")
    ap.add_argument("--out", default="lenet_imc_gpu.json")
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float64 if args.fp64 else torch.float32
    print(f"device: {device}, dtype: {dtype}")

    model = load_model(args.src_dir, args.v3_dir)
    tm = to_torch_model(model, device=device, dtype=dtype)
    times = [float(x) for x in args.drift_times.split(",")]

    if args.mnist:
        sc = calibrate_input_scale(args.mnist_root, args.n_calib)
        print(f"MNIST: calibrated input scale {sc:.12f} (scales.txt rounded {read_input_scale(args.src_dir)})")
        X, y = load_images_mnist(args.n_images, sc, args.mnist_root,
                                 selfcheck_hex=os.path.join(args.src_dir, "images", "img_001.hex"))
    else:
        X, y = load_images_hex(args.hex_dir, os.path.join(args.src_dir, "image_labels.txt"), args.n_images)
    print(f"images: {len(X)}, seeds: {args.seeds}, drift times: {times}")

    arms_np = {"algo_only": {L: np.zeros_like(model["mask"][L], dtype=bool) for L in LAYERS},
               "uniform": make_uniform_mask(model),
               "layer_aware": model["mask"]}
    arms = {k: masks_to_torch(v, device) for k, v in arms_np.items()}

    use_wn = not args.no_write_noise
    nus = [nu_torch(model, 1000 + s, device, dtype) for s in range(args.seeds)]
    wns = [writenoise_torch(model, 5000 + s, device, dtype) if use_wn else None
           for s in range(args.seeds)]
    sc_cell = writenoise_sigma_cell()
    print(f"write noise: {'ON' if use_wn else 'OFF'}"
          + (f" (sigma_cell={sc_cell:.6f}, value-level sigma=3.81 per Paper 1)" if use_wn else ""))

    clean = run_arm(tm, arms["layer_aware"], X, y, nus[0], None, 1.0, 0.0, None, args.chunk)
    print(f"clean (INT8, no drift, no ADC, no write noise): {100*clean:.2f}%")

    results = {"meta": {"n_images": len(X), "seeds": args.seeds, "times": times,
                        "adc_bits": args.adc_bits, "clean": clean, "device": device,
                        "dtype": str(dtype), "sign_scheme": "signmag_cellBit2", "cellBit": 2,
                        "write_noise": use_wn, "write_noise_sigma_cell": (sc_cell if use_wn else 0.0)},
               "data": {}}
    t_start = time.time()
    for arm, mask_t in arms.items():
        for adc_name, adc in [("noADC", None), (f"{args.adc_bits}b", args.adc_bits)]:
            for t in times:
                accs = []
                for s in range(args.seeds):
                    accs.append(run_arm(tm, mask_t, X, y, nus[s], wns[s], t, COMP, adc, args.chunk))
                accs = np.array(accs)
                results["data"][f"{arm}|{adc_name}|{t:.0e}"] = {
                    "mean": float(accs.mean()), "std": float(accs.std()), "vals": accs.tolist()}
                print(f"  {arm:12} {adc_name:6} t={t:.0e}: {100*accs.mean():6.2f}% +/- {100*accs.std():.2f}   "
                      f"[{time.time()-t_start:.0f}s]")

    with open(args.out, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nsaved -> {args.out}")
    print("\n=== layer_aware - uniform gap (pp) ===")
    print(f"  {'t':>7} | {'no-ADC':>8} | {f'{args.adc_bits}-bit':>8}")
    for t in times:
        g0 = 100 * (results["data"][f"layer_aware|noADC|{t:.0e}"]["mean"]
                    - results["data"][f"uniform|noADC|{t:.0e}"]["mean"])
        g5 = 100 * (results["data"][f"layer_aware|{args.adc_bits}b|{t:.0e}"]["mean"]
                    - results["data"][f"uniform|{args.adc_bits}b|{t:.0e}"]["mean"])
        print(f"  {t:7.0e} | {g0:7.2f}  | {g5:7.2f}")


if __name__ == "__main__":
    main()


Writing run_lenet_gpu.py


## 3. (sanity) confirm the files exist

In [6]:
import os
print([f for f in ['imc_vec.py','imc_torch.py','run_lenet_gpu.py'] if os.path.exists(f)])

['imc_vec.py', 'imc_torch.py', 'run_lenet_gpu.py']


## 4. Upload data.zip and unzip
Run this cell, pick `data.zip` (zip of `rtl_inputs_A` + `rtl_inputs_A_v3`, both at the top of the zip).

In [8]:
from google.colab import files
up = files.upload()
!unzip -o data.zip -d . > /dev/null
import os
print('rtl_inputs_A:', os.path.isdir('rtl_inputs_A'),
      '| rtl_inputs_A_v3:', os.path.isdir('rtl_inputs_A_v3'),
      '| images/:', os.path.isdir('rtl_inputs_A/images'))

Saving data.zip to data.zip
rtl_inputs_A: True | rtl_inputs_A_v3: True | images/: True


## 5a. Quick check (2000 images, 2 seeds) - a few minutes
Confirm `input self-check: PASS` and that the gap table trend is there before the full run.

In [9]:
!python run_lenet_gpu.py --src-dir rtl_inputs_A --v3-dir rtl_inputs_A_v3 \
    --mnist --mnist-root ./_mnist --n-images 2000 --seeds 2 --out lenet_imc_quick.json

device: cuda, dtype: torch.float32
100% 9.91M/9.91M [00:00<00:00, 17.8MB/s]
100% 28.9k/28.9k [00:00<00:00, 468kB/s]
100% 1.65M/1.65M [00:00<00:00, 4.39MB/s]
100% 4.54k/4.54k [00:00<00:00, 22.8MB/s]
MNIST: calibrated input scale 0.022216430799 (scales.txt rounded 0.022216)
  input self-check (test[0] vs img_001.hex): PASS
images: 2000, seeds: 2, drift times: [1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0, 1000000.0]
write noise: ON (sigma_cell=0.057641, value-level sigma=3.81 per Paper 1)
clean (INT8, no drift, no ADC, no write noise): 98.30%
  algo_only    noADC  t=1e+00:  98.28% +/- 0.08   [1s]
  algo_only    noADC  t=1e+01:  98.25% +/- 0.10   [3s]
  algo_only    noADC  t=1e+02:  98.30% +/- 0.10   [5s]
  algo_only    noADC  t=1e+03:  98.03% +/- 0.13   [6s]
  algo_only    noADC  t=1e+04:  97.90% +/- 0.50   [7s]
  algo_only    noADC  t=1e+05:  97.47% +/- 0.48   [9s]
  algo_only    noADC  t=1e+06:  96.80% +/- 0.70   [10s]
  algo_only    5b     t=1e+00:  88.90% +/- 0.55   [13s]
  algo_only 

## 5b. Full run (all 10000 images, 5 seeds)
Run after 5a looks right. On a T4 roughly 10-20 minutes.

In [10]:
!python run_lenet_gpu.py --src-dir rtl_inputs_A --v3-dir rtl_inputs_A_v3 \
    --mnist --mnist-root ./_mnist --n-images 0 --seeds 5 --out lenet_imc_full.json

device: cuda, dtype: torch.float32
MNIST: calibrated input scale 0.022216430799 (scales.txt rounded 0.022216)
  input self-check (test[0] vs img_001.hex): PASS
images: 10000, seeds: 5, drift times: [1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0, 1000000.0]
write noise: ON (sigma_cell=0.057641, value-level sigma=3.81 per Paper 1)
clean (INT8, no drift, no ADC, no write noise): 98.92%
  algo_only    noADC  t=1e+00:  98.75% +/- 0.08   [18s]
  algo_only    noADC  t=1e+01:  98.75% +/- 0.10   [37s]
  algo_only    noADC  t=1e+02:  98.66% +/- 0.15   [55s]
  algo_only    noADC  t=1e+03:  98.49% +/- 0.21   [73s]
  algo_only    noADC  t=1e+04:  98.13% +/- 0.37   [92s]
  algo_only    noADC  t=1e+05:  97.54% +/- 0.51   [110s]
  algo_only    noADC  t=1e+06:  96.41% +/- 1.20   [128s]
  algo_only    5b     t=1e+00:  91.11% +/- 0.62   [157s]
  algo_only    5b     t=1e+01:  87.76% +/- 1.57   [185s]
  algo_only    5b     t=1e+02:  78.68% +/- 2.03   [214s]
  algo_only    5b     t=1e+03:  52.17% +/- 6.08   [

## 6. Download the results

In [11]:
from google.colab import files
files.download('lenet_imc_full.json')   # or lenet_imc_quick.json

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>